In [1]:
import os
os.environ["JAVA_HOME"] = "C:\\Program Files\\Eclipse Adoptium\\jdk-8.0.482.8-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "\\bin;" + os.environ["PATH"]

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()
sc = spark.sparkContext

In [2]:
import multiprocessing
print(multiprocessing.cpu_count())

16


In [3]:
%%time
df_matches = spark.read.options(delimiter=",", header=True, inferSchema=True).csv("battlesStaging_12282020_WL_tagged.csv")

CPU times: total: 0 ns
Wall time: 13 s


In [4]:
%%time
df_cards = spark.read.options(delimiter=",", header=True, inferSchema=True).csv("CardMasterListSeason18_12082020.csv")

CPU times: total: 15.6 ms
Wall time: 310 ms


In [5]:
df_matches.createOrReplaceTempView("matches")
df_cards.createOrReplaceTempView("cards")

df_result = spark.sql("""
    SELECT m.*,
        w1.`team.card1.name` as `winner.card1.name`,
        w2.`team.card1.name` as `winner.card2.name`,
        w3.`team.card1.name` as `winner.card3.name`,
        w4.`team.card1.name` as `winner.card4.name`,
        w5.`team.card1.name` as `winner.card5.name`,
        w6.`team.card1.name` as `winner.card6.name`,
        w7.`team.card1.name` as `winner.card7.name`,
        w8.`team.card1.name` as `winner.card8.name`,
        l1.`team.card1.name` as `loser.card1.name`,
        l2.`team.card1.name` as `loser.card2.name`,
        l3.`team.card1.name` as `loser.card3.name`,
        l4.`team.card1.name` as `loser.card4.name`,
        l5.`team.card1.name` as `loser.card5.name`,
        l6.`team.card1.name` as `loser.card6.name`,
        l7.`team.card1.name` as `loser.card7.name`,
        l8.`team.card1.name` as `loser.card8.name`
    FROM matches m
    LEFT JOIN cards w1 ON m.`winner.card1.id` = w1.`team.card1.id`
    LEFT JOIN cards w2 ON m.`winner.card2.id` = w2.`team.card1.id`
    LEFT JOIN cards w3 ON m.`winner.card3.id` = w3.`team.card1.id`
    LEFT JOIN cards w4 ON m.`winner.card4.id` = w4.`team.card1.id`
    LEFT JOIN cards w5 ON m.`winner.card5.id` = w5.`team.card1.id`
    LEFT JOIN cards w6 ON m.`winner.card6.id` = w6.`team.card1.id`
    LEFT JOIN cards w7 ON m.`winner.card7.id` = w7.`team.card1.id`
    LEFT JOIN cards w8 ON m.`winner.card8.id` = w8.`team.card1.id`
    LEFT JOIN cards l1 ON m.`loser.card1.id` = l1.`team.card1.id`
    LEFT JOIN cards l2 ON m.`loser.card2.id` = l2.`team.card1.id`
    LEFT JOIN cards l3 ON m.`loser.card3.id` = l3.`team.card1.id`
    LEFT JOIN cards l4 ON m.`loser.card4.id` = l4.`team.card1.id`
    LEFT JOIN cards l5 ON m.`loser.card5.id` = l5.`team.card1.id`
    LEFT JOIN cards l6 ON m.`loser.card6.id` = l6.`team.card1.id`
    LEFT JOIN cards l7 ON m.`loser.card7.id` = l7.`team.card1.id`
    LEFT JOIN cards l8 ON m.`loser.card8.id` = l8.`team.card1.id`
""")

df_result.cache()
print("Done!", df_result.count(), "rows")
df_result.show(1)

Done! 1902766 rows
+---+-------------------+----------+-----------+------------------------+----------+-----------------------+-------------------+-------------+-------------------------+------------------------------+---------------+-------------------+----------+----------------------+------------------+------------+------------------------+--------------+------------------+-----------------------------+-------------+---------------+------------------+---------------+------------------+---------------+------------------+---------------+------------------+---------------+------------------+---------------+------------------+---------------+------------------+---------------+------------------+--------------------+----------------------+------------------+----------------------+------------------+-------------------+-----------------+-----------------+----------------------+---------------------+--------------+-----------------+--------------+-----------------+--------------+----------

In [7]:
%%time
from pyspark.sql.functions import col, lit, sum as spark_sum

# Build a union of all winner and loser card columns
card_dfs = []

for i in range(1, 9):
    card_dfs.append(
        df_result.select(col(f"`winner.card{i}.name`").alias("card_name"), lit(1).alias("wins"), lit(0).alias("losses"))
    )
    card_dfs.append(
        df_result.select(col(f"`loser.card{i}.name`").alias("card_name"), lit(0).alias("wins"), lit(1).alias("losses"))
    )

# Union all together
from functools import reduce
from pyspark.sql import DataFrame

all_cards = reduce(DataFrame.union, card_dfs)

# Filter nulls, group and aggregate
df_card_stats = all_cards \
    .filter(col("card_name").isNotNull()) \
    .groupBy("card_name") \
    .agg(
        spark_sum("wins").alias("win_count"),
        spark_sum("losses").alias("loss_count")
    ) \
    .orderBy(col("win_count").desc())

df_card_stats.cache()
df_card_stats.show(10)

+-------------+---------+----------+
|    card_name|win_count|loss_count|
+-------------+---------+----------+
|       Wizard|   582969|    583754|
|     Valkyrie|   570154|    558447|
|      The Log|   542295|    531042|
|          Zap|   524471|    501257|
|Skeleton Army|   517161|    510175|
|     Fireball|   477821|    485420|
|    Hog Rider|   413462|    399929|
|       Arrows|   397171|    394434|
|  Mega Knight|   388330|    387939|
|  Baby Dragon|   360764|    355725|
+-------------+---------+----------+
only showing top 10 rows

CPU times: total: 31.2 ms
Wall time: 17.1 s


In [8]:
%%time
from pyspark.sql.functions import round as spark_round

df_card_stats = df_card_stats.withColumn(
    "win_ratio",
    spark_round(col("win_count") / (col("win_count") + col("loss_count")), 3)
)

df_card_stats.cache()
df_card_stats.show(10)

+-------------+---------+----------+---------+
|    card_name|win_count|loss_count|win_ratio|
+-------------+---------+----------+---------+
|       Wizard|   582969|    583754|      0.5|
|     Valkyrie|   570154|    558447|    0.505|
|      The Log|   542295|    531042|    0.505|
|          Zap|   524471|    501257|    0.511|
|Skeleton Army|   517161|    510175|    0.503|
|     Fireball|   477821|    485420|    0.496|
|    Hog Rider|   413462|    399929|    0.508|
|       Arrows|   397171|    394434|    0.502|
|  Mega Knight|   388330|    387939|      0.5|
|  Baby Dragon|   360764|    355725|    0.504|
+-------------+---------+----------+---------+
only showing top 10 rows

CPU times: total: 0 ns
Wall time: 1.86 s


In [ ]:
%%time
df_card_stats.orderBy(col("win_ratio").desc()).show(df_card_stats.count(), truncate=False)